In [1]:
from structures.card import Carta, Pokémon, Trainer, Energy
from structures.deck import Deck
from structures.expansion import Expansion
from structures.similarity import Similarity

from pathlib import Path
import json, traceback
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm 

from __future__ import annotations
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Iterable, Tuple, List
from itertools import chain
from tqdm import tqdm

from scipy.sparse import save_npz, csr_matrix
import numpy as np
import json


DECKS_PATH = Path('data/tournament_decks')
OUTPUT_PATH = Path("cache/decks.json")



Definiamo un paio di funzioni per caricare tutti i file in una directory oppure solo una lista ristretta. 
da questi file viene estratto un numero variabile di mazzi (Deck). 
Se sono presenti errori, il mazzo viene scartato

In [ ]:
def process_file(file_path):
	"""
	Carica un singolo file JSON e ne estrae i deck.
	Ritorna (decks, errore): decks è lista di Deck; errore è dict o None.
	"""
	try:
		with file_path.open('r', encoding='utf-8') as f:
			data = json.load(f)
		decks = Deck.from_tournament_data(data)
		return decks, None
	except Exception as e:
		return [], {
			"file": str(file_path),
			"errore": str(e),
			"traceback": traceback.format_exc()
		}

def _normalize_paths(paths: Path | Iterable[Path], pattern: str, recursive: bool) -> List[Path]:
	# Accetta: directory, file, lista/iterabile
	if isinstance(paths, Path):
		if paths.is_dir():
			return sorted(paths.rglob(pattern) if recursive else paths.glob(pattern))
		else:
			return [paths] if paths.suffix.lower() == ".json" else []
	# Iterabile di Path
	ps = list(paths)
	return [p for p in ps if p.is_file() and p.suffix.lower() == ".json"]

def load_decks(
		paths: Path | Iterable[Path],
		pattern: str = "*.json",
		recursive: bool = False,
		max_workers: int = 10,
		strict: bool = False
		# count: int = 0
	) -> Tuple[List["Deck"], List[dict]]:
	"""
	Carica deck da più file JSON in parallelo.

	- paths: directory, file singolo o iterabile di Path
	- pattern: glob (usato se 'paths' è una directory)
	- recursive: usa rglob sul pattern
	- max_workers: thread pool size
	- strict: se True, solleva al primo errore (per pipeline CI)
	"""
	file_list = _normalize_paths(paths, pattern, recursive)
	if not file_list:
		return [], []

	workers = max(1, min(max_workers, len(file_list)))
	chunksize = max(1, len(file_list) // (workers * 4))

	decks: List["Deck"] = []
	errori: List[dict] = []

	iter_results = None
	with ThreadPoolExecutor(max_workers=workers) as executor:
		iter_results = executor.map(process_file, file_list, chunksize=chunksize)

		iter_results = tqdm(iter_results, total=len(file_list), unit="file",
		                    desc="Caricamento JSON")

		for result_decks, err in iter_results:
			for deck in result_decks:
				if deck.len() > 0:
					decks.append(deck)
			if err:
				if strict:
					# Mostra il contesto e interrompe
					raise RuntimeError(f"Errore su {err.get('file')}: {err.get('errore')}")
				errori.append(err)
			# if count != 0 and len(decks) >= count:
			# 	break
			

	return decks, errori

Carichiamo tramite threading tutti i mazzi presenti in DECKSPATH
Questo processo è lungo e tedioso, per fortuna va fatto solo una volta

# Estrapolazione dei dati, salvataggio, caricamento, visualizzazione
#### Estrapola

In [ ]:
# Decommenta queste linee sotto per estrapolare solo i primi n file
# n=1
# DECKS_PATH = [p for p in (DECKS_PATH.glob("*.json"))][:n]

# Decommenta queste per caricare i mazzi
# decks, errori = load_decks(DECKS_PATH, max_workers=10)
# print(f"Decks caricati: {len(decks)}")

`encyclopedia` è il dizionario completo di tutte le carte, usando il metodo `card.to_dict()`. 

Racchiude solo le carte presenti nei mazzi, e solo la prima occorrenza

In [ ]:
encyclopedia: dict[str, dict] = {}
for deck in decks:
    for carta, qty in deck.carte.items():
        encyclopedia[carta.get_id()] = carta.to_dict()



Creiamo X, una matrice **CSR** di shape *n_decks x n_cards* con i conteggi di ogni carta presente in ogni mazzo.
- **deck_ids**: np.ndarray[str] che identifica l'ordine delle righe, costruito sugli *uid* dei mazzi
- **card_ids**: np.ndarray[str] che identifica l'ordine delle colonne, costruito sugli *id* delle carte
- X: csr_matrix uint16 (conteggi)

In [ ]:
card_ids = np.array(sorted(encyclopedia.keys()), dtype=object)
deck_ids = np.array([d.uid() for d in decks], dtype=object)
card_to_col = {cid: i for i, cid in enumerate(card_ids)}


indptr = [0]
indices: list[int] = []
data: list[int] = []

for d in decks:
    for carta, qty in d.carte.items():
        col = card_to_col[carta.get_id()]
        indices.append(col)
        data.append(int(qty))
    indptr.append(len(indices))
    
# Matrice CSR
X = csr_matrix(
    (np.asarray(data, dtype=np.uint16),
        np.asarray(indices, dtype=np.int32),
        np.asarray(indptr, dtype=np.int64)),
    shape=(len(decks), len(card_ids))
)

#### Salva

In [ ]:
# Salva tutti i mazzi in json
# with OUTPUT_PATH.open("w", encoding="utf-8") as f:
#     json.dump([deck.to_dict() for deck in decks], f, ensure_ascii=False, indent=2)

In [ ]:
with Path("cache/cards.json").open("w", encoding="utf-8") as f:
    json.dump(encyclopedia, f, ensure_ascii=False, indent=2)

In [ ]:
def csr_save(X: csr_matrix, deck_ids, card_ids, path="cache/decks_csr.npz"):
	np.savez(
		path,
		data=X.data,
		indices=X.indices,
		indptr=X.indptr,
		shape=np.array(X.shape),
		deck_ids=deck_ids,
		card_ids=card_ids,
	)

csr_save(X, deck_ids, card_ids)

#### Carica

In [ ]:
# Caricamento lento (20 minuti) di tutti i deck da deck.json

# with OUTPUT_PATH.open("r", encoding="utf-8") as f:
#     data = json.load(f)
# decks = [Deck.from_dict(d) for d in data]

In [2]:
def csr_load(path: str="cache/decks_csr.npz"):
	z = np.load(path, allow_pickle=True)
	X = csr_matrix(
		(z["data"], z["indices"], z["indptr"]),
		shape=tuple(z["shape"])
	)
	return X, z["deck_ids"], z["card_ids"]

X, deck_ids, card_ids = csr_load()

In [3]:
# Caricamento di tutte le carte
with open("cache/cards.json", encoding="utf-8") as f:
    encyclopedia = json.load(f)

#### Visualizza

In [ ]:
# Stampa la rappresentazione dizionario di una carta dall'enciclopedia
encyclopedia["swsh8-254"]

{'id': 'swsh8-254',
 'name': 'Genesect V',
 'supertype': 'Pokémon',
 'subtypes': ['Basic', 'V', 'Fusion Strike'],
 'expansion': 'swsh8',
 'number': '254',
 'type': 'Colorless',
 'evolves_from': None,
 'evolves_to': [],
 'weaknesses': ['Fire'],
 'resistances': ['Grass'],
 'hp': 190,
 'attacks': [{'name': 'Techno Blast',
   'text': "During your next turn, this Pokémon can't attack.",
   'damage': 210,
   'convertedEnergyCost': 3}],
 'abilities': [{'name': 'Fusion Strike System',
   'text': 'Once during your turn, you may draw cards until you have as many cards in your hand as you have Fusion Strike Pokémon in play.'}],
 'retreat_cost': 0}

In [ ]:
# Stampa le carte di un mazzo e le loro quantità
row = X.getrow(0)
lines = [f"{cid}\t{int(q)}" for cid, q in zip(card_ids[row.indices], row.data)]
print("\n".join(lines[:6]) + "\n...")


swsh8-254	4
swsh8-124	2
swsh8-113	4
swsh8-114	3
swsh5-125	2
swsh8-236	4
...


In [ ]:
# Crea una carta dall'enciclopedia
Carta.from_dict(encyclopedia["swsh8-254"])

(swsh8-254)	Genesect V

In [ ]:
# Crea un mazzo dalla sparse_row
Deck.from_sparse_row(
	X.getrow(0), 
	card_ids=card_ids, 
	cards_encyclopedia=encyclopedia,
	name=str(deck_ids[0])
)


<Deck '427cc15408ba7ec46b00b4c6bfc8d304', 60 carte [swsh8-254×4, swsh8-124×2, swsh8-113×4, swsh8-114×3, swsh5-125×2 ", ..." ]